# SpaceX Falcon 9 - EDA with SQL

Load the SpaceX launch table into an in-memory SQLite database and answer questions with SQL.

In [ ]:
import sqlite3
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

In [ ]:
df = pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0321EN-SkillsNetwork/labs/module_2/data/Spacex.csv')
df = df.rename(columns={'Time (UTC)': 'Time'})
df.columns = [col.strip() for col in df.columns]
print(list(df.columns))
df.head()

In [ ]:
# Date format in the source is DD-MM-YYYY; add an ISO date column for easy filtering
df['Date_ISO'] = pd.to_datetime(df['Date'], dayfirst=True, errors='coerce').dt.strftime('%Y-%m-%d')
con = sqlite3.connect(':memory:')
df.to_sql('SPACEXTBL', con, index=False, if_exists='replace')

def q(sql):
    return pd.read_sql_query(sql, con)

q('SELECT COUNT(*) AS total_rows FROM SPACEXTBL')

### Task 1: Names of the unique launch sites

In [ ]:
q('SELECT DISTINCT Launch_Site FROM SPACEXTBL')

### Task 2: 5 records where the launch site begins with 'CCA'

In [ ]:
q("SELECT * FROM SPACEXTBL WHERE Launch_Site LIKE 'CCA%' LIMIT 5")

### Task 3: Total payload mass carried by boosters launched by NASA (CRS)

In [ ]:
q("SELECT SUM(PAYLOAD_MASS__KG_) AS total_payload_kg FROM SPACEXTBL WHERE Customer LIKE 'NASA (CRS)%'")

### Task 4: Average payload mass carried by booster version F9 v1.1

In [ ]:
q("SELECT AVG(PAYLOAD_MASS__KG_) AS avg_payload_kg FROM SPACEXTBL WHERE Booster_Version LIKE 'F9 v1.1%'")

### Task 5: Date of the first successful landing outcome on a ground pad

In [ ]:
q("SELECT MIN(Date_ISO) AS first_ground_pad_success FROM SPACEXTBL WHERE Landing_Outcome LIKE 'Success (ground pad)%'")

### Task 6: Boosters with success on a drone ship and payload mass between 4000 and 6000 kg

In [ ]:
q("SELECT DISTINCT Booster_Version FROM SPACEXTBL WHERE Landing_Outcome LIKE 'Success (drone ship)%' AND PAYLOAD_MASS__KG_ > 4000 AND PAYLOAD_MASS__KG_ < 6000")

### Task 7: Total number of successful and failed mission outcomes

In [ ]:
q('SELECT Mission_Outcome, COUNT(*) AS total FROM SPACEXTBL GROUP BY Mission_Outcome')

### Task 8: Booster versions that carried the maximum payload mass

In [ ]:
q('SELECT DISTINCT Booster_Version, PAYLOAD_MASS__KG_ FROM SPACEXTBL WHERE PAYLOAD_MASS__KG_ = (SELECT MAX(PAYLOAD_MASS__KG_) FROM SPACEXTBL)')

### Task 9: Failed drone ship landings in 2015 (month, booster version, launch site)

In [ ]:
q("SELECT substr(Date_ISO, 6, 2) AS month, Landing_Outcome, Booster_Version, Launch_Site FROM SPACEXTBL WHERE Landing_Outcome LIKE 'Failure (drone ship)%' AND substr(Date_ISO, 1, 4) = '2015'")

### Task 10: Rank landing outcomes between 2010-06-04 and 2017-03-20 (descending count)

In [ ]:
q("SELECT Landing_Outcome, COUNT(*) AS outcome_count FROM SPACEXTBL WHERE Date_ISO BETWEEN '2010-06-04' AND '2017-03-20' GROUP BY Landing_Outcome ORDER BY outcome_count DESC")